# 🚢 Maritime Route Optimizer — Kepler.gl Visualization

Interactive visualization of:
- Real AIS vessel trajectories
- Port graph (nodes + edges)
- Optimized routes with GNN-predicted costs

**Output:** `app/maritime_routes.html` — standalone interactive map

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from keplergl import KeplerGl
import sys
sys.path.append('..')

print('✅ Imports OK')

✅ Imports OK


## 1. Load Data

In [2]:
# Load clean AIS trajectories (sample for performance)
ais = pd.read_parquet('../data/processed/ais_features.parquet')

# Sample 50k points for smooth rendering
ais_sample = ais.sample(50_000, random_state=42).copy()
ais_sample['timestamp'] = ais_sample['timestamp'].astype(str)

print(f'AIS sample: {len(ais_sample):,} points')
print(f'Columns: {list(ais_sample.columns)}')

AIS sample: 50,000 points
Columns: ['mmsi', 'timestamp', 'vessel_name', 'vessel_category', 'lat', 'lon', 'sog', 'cog', 'distance_km', 'time_delta_s', 'speed_change', 'heading_change', 'hour_of_day', 'day_of_week', 'is_weekend', 'month', 'length', 'width', 'draft', 'avg_speed', 'std_speed', 'total_distance_km', 'n_points']


In [3]:
# Load port graph
nodes = pd.read_parquet('../data/processed/graph_nodes.parquet')
edges = pd.read_parquet('../data/processed/graph_edges.parquet')

print(f'Ports (nodes): {len(nodes)}')
print(f'Routes (edges): {len(edges)}')
nodes.head()

Ports (nodes): 67
Routes (edges): 63


,wpi_number,name,country,lat,lon,water_body,port_id
0,9210.0,Norsworthy,United States,29.733333,-95.200000,Gulf of Mexico; North Atlantic Ocean,10
1,9240.0,Houston,United States,29.750000,-95.283333,Gulf of Mexico; North Atlantic Ocean,77
2,8280.0,Norfolk,United States,36.850000,-76.300000,North Atlantic Ocean,128
3,7460.0,Greenport,United States,41.100000,-72.366667,North Atlantic Ocean,130
4,16310.0,Yerbabuena Island,United States,37.816667,-122.366667,North Pacific Ocean,167


## 2. Prepare Layers

In [4]:
# Layer 1: AIS trajectories
trajectories = ais_sample[['mmsi', 'lat', 'lon', 'sog', 'vessel_category', 'timestamp']].copy()
trajectories = trajectories.rename(columns={'sog': 'speed_knots'})
print('Layer 1 - Trajectories:', trajectories.shape)

Layer 1 - Trajectories: (50000, 6)


In [5]:
# Layer 2: Port nodes
ports_layer = nodes[['name', 'country', 'lat', 'lon']].copy()
print('Layer 2 - Ports:', ports_layer.shape)

Layer 2 - Ports: (67, 4)


In [6]:
# Layer 3: Route edges (arc layer)
edges_layer = edges.merge(
    nodes[['port_id', 'lat', 'lon', 'name']].rename(
        columns={'port_id': 'departure_port_id', 'lat': 'src_lat', 'lon': 'src_lon', 'name': 'src_name'}
    ),
    on='departure_port_id', how='left'
).merge(
    nodes[['port_id', 'lat', 'lon', 'name']].rename(
        columns={'port_id': 'arrival_port_id', 'lat': 'dst_lat', 'lon': 'dst_lon', 'name': 'dst_name'}
    ),
    on='arrival_port_id', how='left'
)
edges_layer = edges_layer[[
    'src_name', 'dst_name', 'src_lat', 'src_lon',
    'dst_lat', 'dst_lon', 'n_vessels', 'avg_speed', 'avg_distance_km'
]].dropna()
print('Layer 3 - Edges:', edges_layer.shape)

Layer 3 - Edges: (63, 9)


In [7]:
# Layer 4: Optimized routes via GNN + A*
from src.models.optimizer import MaritimeRouteOptimizer

opt = MaritimeRouteOptimizer(
    nodes_path='../data/processed/graph_nodes.parquet',
    edges_path='../data/processed/graph_edges.parquet',
    model_path='../data/processed/gnn_model.pt',
)

route_pairs = [
    ('Los Angeles', 'Long Beach'),
    ('New Orleans', 'Gretna'),
    ('Norfolk', 'Newport News'),
    ('Houston', 'Baytown'),
    ('San Francisco', 'Oakland'),
]

optimized_routes = []
for origin, dest in route_pairs:
    result = opt.optimize(origin, dest)
    if result.found:
        for i, (lat, lon) in enumerate(result.path_coords):
            optimized_routes.append({
                'route': f'{result.origin} → {result.destination}',
                'step': i,
                'lat': lat,
                'lon': lon,
                'port_name': result.path_ports[i],
                'total_cost': result.total_cost,
                'total_distance_km': result.total_distance_km,
            })

routes_layer = pd.DataFrame(optimized_routes)
print(f'Layer 4 - Optimized routes: {len(routes_layer)} points, {routes_layer["route"].nunique()} routes')

2026-03-11 21:21:42,714 [INFO] Using device: mps
2026-03-11 21:21:42,715 [INFO] Initializing Maritime Route Optimizer...
2026-03-11 21:21:42,718 [INFO] Building PyG graph: 67 nodes, 63 edges
2026-03-11 21:21:42,725 [INFO]   Node features : torch.Size([67, 4])
2026-03-11 21:21:42,725 [INFO]   Edge index    : torch.Size([2, 63])
2026-03-11 21:21:42,725 [INFO]   Edge features : torch.Size([63, 3])
2026-03-11 21:21:43,042 [INFO]   Ports available : 67
2026-03-11 21:21:43,043 [INFO]   Routes available: 63
2026-03-11 21:21:43,043 [INFO] ✅ Optimizer ready!
2026-03-11 21:21:43,045 [INFO] 
Optimizing route: Los Angeles → Long Beach
2026-03-11 21:21:43,045 [INFO]   Path    : Los Angeles → Long Beach
2026-03-11 21:21:43,045 [INFO]   Hops    : 1
2026-03-11 21:21:43,046 [INFO]   Cost    : 0.1676
2026-03-11 21:21:43,046 [INFO]   Distance: 6.4 km
2026-03-11 21:21:43,047 [INFO] 
Optimizing route: New Orleans → Gretna
2026-03-11 21:21:43,048 [INFO]   Path    : New Orleans → Gretna
2026-03-11 21:21:43,0

Layer 4 - Optimized routes: 6 points, 3 routes


## 3. Build & Display Map

In [9]:
config = {
    'version': 'v1',
    'config': {
        'mapState': {
            'latitude': 32.0, 'longitude': -95.0,
            'zoom': 4, 'bearing': 0, 'pitch': 0,
        },
        #'mapStyle': {'styleType': 'dark'},
    }
}

map_ = KeplerGl(height=700, config=config)
map_.add_data(data=trajectories, name='AIS Trajectories')
map_.add_data(data=ports_layer, name='Port Nodes')
map_.add_data(data=edges_layer, name='Route Edges')
map_.add_data(data=routes_layer, name='Optimized Routes')

map_

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(config={'version': 'v1', 'config': {'mapState': {'latitude': 32.0, 'longitude': -95.0, 'zoom': 4, 'be…

## 4. Export to HTML

In [10]:
output_path = '../app/maritime_routes.html'
Path('../app').mkdir(exist_ok=True)
map_.save_to_html(file_name=output_path)
print(f'✅ Map saved to {output_path}')
print(f'   Open: file://{Path(output_path).resolve()}')

Map saved to ../app/maritime_routes.html!
✅ Map saved to ../app/maritime_routes.html
   Open: file:///Users/naim/Desktop/Ex_OC_nonlocalhost/maritime-route-optimizer/app/maritime_routes.html
